重要属性ambience组合过多，不适合作为结构化属性，而是合并进item的文本描述

RestaurantsPriceRange2 [1,2,3,4] （防止增加维度放弃one-hot，选择归一化+descripition）

其余属性都是binary

✅ **Extract essential metadata**:

- Retain key identifiers and basic details:
    - `"business_id"`, `"name"`, `"stars"`, `"review_count"`

✅ **Process `RestaurantsPriceRange2`**:

- Convert price range (`1-4`) into **text descriptions**:
    - `"budget-friendly", "moderate", "expensive", "high-end"`
- (Optional for later) **Normalize price** for structured data representation.

✅ **Process `Ambience`**:

- Convert **structured ambience attributes** into a **text description**
    - Example: `"casual, romantic, trendy"` → `"with casual, romantic, and trendy ambience"`
- If `Ambience` is `"Unknown"`, exclude from the description.

✅ **Filter `categories`**:

- Retain **only valid categories** from `pa_cleaned_categories_2.json`.
- Assign each business to a **category cluster** based on:
    - Categories appearing in **hybrid clustering results**.
    - Most frequently occurring cluster among associated categories.

✅ **Convert binary attributes to standalone fields**:

- `"RestaurantsTakeOut", "BusinessParking", "BikeParking", "RestaurantsDelivery", "RestaurantsReservations"`
- Convert from `"True"/"False"` to **`1/0` representation**.

✅ **Construct a structured `description`**:

- Format dynamically based on available attributes:
    - `"Restaurant Name: a budget-friendly restaurant with casual ambience categorized under Italian, Pizza"`
    - If no price or ambience is available, exclude those parts.

In [19]:
import os
import json

# Set file paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
input_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_3.json")
output_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_4.json")
category_cluster_path = os.path.join(BASE_DIR, "output_categories", "category_clusters_hybrid.json")
cleaned_categories_path = os.path.join(BASE_DIR, "output_categories", "pa_cleaned_categories_2.json")

# Load business data
with open(input_path, "r", encoding="utf-8") as f:
    businesses = json.load(f)

# Load category cluster mapping
with open(category_cluster_path, "r", encoding="utf-8") as f:
    category_clusters = json.load(f)

# Load cleaned categories (only valid categories should be retained)
with open(cleaned_categories_path, "r", encoding="utf-8") as f:
    valid_categories = set(json.load(f))  # Convert to set for fast lookup

# Reverse lookup: {category: cluster_id}
category_to_cluster = {}
for cluster_id, category_list in category_clusters.items():
    for category in category_list:
        category_to_cluster[category] = cluster_id

# Mapping for price range (for description only)
PRICE_TEXT_MAP = {
    "1": "budget-friendly",  
    "2": "moderately priced",         
    "3": "expensive",        
    "4": "high-end"         
}

processed_businesses = []  # Store cleaned business data

# Process each business entry
for business in businesses:
    attributes = business.get("attributes", {})

    # Extract essential fields
    processed_entry = {
        "business_id": business.get("business_id", ""),
        "name": business.get("name", ""),
        "stars": business.get("stars", 0.0),
        "review_count": business.get("review_count", 0),
    }

    # Retain RestaurantsPriceRange2 without normalization
    processed_entry["RestaurantsPriceRange2"] = attributes.get("RestaurantsPriceRange2", None)

    # Process price text (for description, but do not store normalized price)
    price_text = PRICE_TEXT_MAP.get(attributes.get("RestaurantsPriceRange2"), None)

    # Process Ambience
    ambience = attributes.get("Ambience", None)
    if isinstance(ambience, str) and ambience.lower() != "unknown":
        ambience_text = ambience.replace("_", " ").lower()
    else:
        ambience_text = None

    # Process Categories → Assign Cluster
    categories = business.get("categories", "")
    category_list = [c.strip() for c in categories.split(",")] if categories else []

    # Keep only valid categories
    filtered_categories = [c for c in category_list if c in valid_categories]

    # Assign the most relevant cluster
    cluster_counts = {}
    for cat in filtered_categories:
        cluster_id = category_to_cluster.get(cat, None)
        if cluster_id:
            cluster_counts[cluster_id] = cluster_counts.get(cluster_id, 0) + 1

    if cluster_counts:
        best_cluster = max(cluster_counts, key=cluster_counts.get)
        processed_entry["category_cluster"] = int(best_cluster)
    else:
        processed_entry["category_cluster"] = None

    # Build dynamic description
    description_parts = []
    if processed_entry["name"]:
        description_parts.append(f"{processed_entry['name']}:")  
    if price_text:
        description_parts.append(f"a {price_text} restaurant")
    if ambience_text:
        description_parts.append(f"with {ambience_text} ambience")
    if filtered_categories:
        description_parts.append("categorised under " + ", ".join(filtered_categories))

    # Update business description
    processed_entry["description"] = " ".join(description_parts) if len(description_parts) > 1 else ""

    # Retain binary attributes in original format (no binarization)
    for attr in ["RestaurantsTakeOut", "BusinessParking", "BikeParking", "RestaurantsDelivery", "RestaurantsReservations"]:
        processed_entry[attr] = attributes.get(attr, "Unknown")  # Keep as 'True', 'False', or 'Unknown' instead of 0/1

    # Add valid categories
    processed_entry["categories"] = filtered_categories

    # Store processed entry
    processed_businesses.append(processed_entry)

# Save processed data
with open(output_path, "w", encoding="utf-8") as f: 
    json.dump(processed_businesses, f, indent=2)  
 
print(f"Updated business dataset saved to: {output_path}")

Updated business dataset saved to: d:\Programming\LLM_RS\output_businesses\pa_valid_filtered_dining_4.json


1. **Counts occurrences** of categorical and binary attributes.
2. **Summarizes unique values** for categorical attributes.
3. **Computes mean, median, min, max, and standard deviation** for numerical attributes.
4. **Saves the statistical summary in JSON format**.

In [21]:
import os
import json
import numpy as np
from collections import Counter, defaultdict

# Set file paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
input_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_4.json")
output_path = os.path.join(BASE_DIR, "output_attributes", "attribute_statistics.json")

# Load cleaned business data
with open(input_path, "r", encoding="utf-8") as f:
    businesses = json.load(f)

# Initialize counters and statistical trackers
attribute_counts = Counter()  # Count occurrences of attributes
attribute_values = defaultdict(Counter)  # Count occurrences of each unique value
numerical_stats = defaultdict(list)  # Store numerical values for statistics

# Identify numerical and categorical attributes
NUMERICAL_ATTRIBUTES = {"stars", "review_count", "RestaurantsPriceRange2"}
CATEGORICAL_ATTRIBUTES = {"category_cluster"}
BINARY_ATTRIBUTES = {"RestaurantsTakeOut", "BusinessParking", "BikeParking", "RestaurantsDelivery", "RestaurantsReservations"}

# Process each business entry
for business in businesses:
    for key, value in business.items():
        if key in NUMERICAL_ATTRIBUTES:
            if value is not None:
                numerical_stats[key].append(float(value))  # Convert to float for calculations
        elif key in CATEGORICAL_ATTRIBUTES or key in BINARY_ATTRIBUTES:
            attribute_counts[key] += 1  # Track total occurrence
            attribute_values[key][value] += 1  # Track value distribution

# Compute numerical attribute statistics
numerical_summary = {
    attr: {
        "count": int(len(values)),
        "mean": float(np.mean(values)) if values else None,
        "median": float(np.median(values)) if values else None,
        "min": float(np.min(values)) if values else None,
        "max": float(np.max(values)) if values else None,
        "std_dev": float(np.std(values)) if values else None
    }
    for attr, values in numerical_stats.items()
}

# Convert defaultdict to normal dictionary
attribute_summary = {
    "counts": dict(attribute_counts),
    "values": {k: dict(v) for k, v in attribute_values.items()},
    "numerical_summary": numerical_summary
}

# Ensure output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save summary to a JSON file (fixing JSON serialization issue)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(attribute_summary, f, indent=2)

# Display summary
print(f"Attribute statistics saved to: {output_path}")


Attribute statistics saved to: d:\Programming\LLM_RS\output_attributes\attribute_statistics.json


- `stars`: Normally distributed (3.0–5.0)
- `review_count`: Skewed (Max = 4,250, Mean = 117.86, Median = 54.5)
- `RestaurantsPriceRange2`: Missing for **15.7%** of businesses
- **Binary Attributes**: `"Unknown"` values exist, need to be converted to `0/1`
- **Category Clusters**: Imbalanced distribution (some overrepresented), but in real-world scenarios, this makes sense

In [24]:
import os
import json
import numpy as np
import pandas as pd

# Set file paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
input_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_4.json")
output_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_5.json")

# Load data
with open(input_path, "r", encoding="utf-8") as f: 
    businesses = json.load(f)  

# Convert to DataFrame  
df = pd.DataFrame(businesses)  

# Impute missing RestaurantsPriceRange2 using cluster median 
df["RestaurantsPriceRange2"] = pd.to_numeric(df["RestaurantsPriceRange2"], errors="coerce")  
df["category_cluster"] = df["category_cluster"].astype("Int64")  

cluster_median_price = df.groupby("category_cluster")["RestaurantsPriceRange2"].median()  
df["RestaurantsPriceRange2"] = df.apply(  
    lambda x: cluster_median_price[x["category_cluster"]] if pd.isna(x["RestaurantsPriceRange2"]) else x["RestaurantsPriceRange2"], 
    axis=1  
) 

# Normalize price range  
df["normalized_price"] = (df["RestaurantsPriceRange2"] - 1) / (4 - 1) 

# Bayesian Rating Adjustment for qual_pop_wt_score 
global_avg_rating = df["stars"].mean()  # Global average rating 
m = 50  # Smoothing factor, controls how much weight is given to small-review businesses 

df["qual_pop_wt_score"] = (df["stars"] * df["review_count"] + m * global_avg_rating) / (df["review_count"] + m) 

# Convert binary attributes to 0/1  
binary_columns = ["RestaurantsTakeOut", "BusinessParking", "BikeParking", "RestaurantsDelivery", "RestaurantsReservations"]  
df[binary_columns] = df[binary_columns].replace({"True": 1, "False": 0, "Unknown": 0}).astype(int)

# Drop unneeded columns
df.drop(columns=["RestaurantsPriceRange2"], inplace=True)

# Save preprocessed data
df.to_json(output_path, orient="records", indent=2)

print(f"Preprocessed dataset saved to: {output_path}")

Preprocessed dataset saved to: d:\Programming\LLM_RS\output_businesses\pa_valid_filtered_dining_5.json


C:\Users\yuqi\AppData\Local\Temp\ipykernel_24028\3297748879.py:39: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_columns] = df[binary_columns].replace({"True": 1, "False": 0, "Unknown": 0}).astype(int)
